# 1. Import thư viện

In [10]:
import pandas as pd
import glob
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix 

# 2. Đọc dữ liệu từ file

In [11]:
path = '../VLSP2018-SA-train-dev-test' 
all_files = glob.glob(os.path.join(path, "*.txt"))

texts, labels = [], []

for filename in all_files:
    with open(filename, 'r', encoding='utf-8-sig') as f:
        # Tách từng cụm (cách nhau bởi dòng trống)
        blocks = f.read().strip().split('\n\n')
        for block in blocks:
            lines = block.split('\n')
            if len(lines) >= 3:
                content = lines[1] # Dòng chữ
                sentiment_line = lines[2].lower() # Dòng nhãn
                
                # Gán nhãn: 2 (Pos), 0 (Neg), 1 (Neu)
                if 'negative' in sentiment_line: label = 0
                elif 'positive' in sentiment_line: label = 2
                else: label = 1
                
                texts.append(content)
                labels.append(label)

# 3. Biến chữ thành số

In [12]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), token_pattern=r'(?u)\b\w\w+\b')
X = vectorizer.fit_transform(texts)
y = labels

# 4. Huấn luyện mô hình Softmax regression

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)
print(f"Đã học xong từ {len(texts)} mẫu dữ liệu!")

C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Đã học xong từ 10351 mẫu dữ liệu!


# 5. Kết quả

In [14]:
print("Độ chính xác chi tiết:")
print(classification_report(y_test, model.predict(X_test), target_names=['Neg', 'Neu', 'Pos']))
y_pred=model.predict(X_test)
print(confusion_matrix(y_test, y_pred))

Độ chính xác chi tiết:
              precision    recall  f1-score   support

         Neg       0.87      0.67      0.76       704
         Neu       0.00      0.00      0.00        38
         Pos       0.83      0.95      0.89      1329

    accuracy                           0.84      2071
   macro avg       0.57      0.54      0.55      2071
weighted avg       0.83      0.84      0.83      2071

[[ 474    0  230]
 [   5    0   33]
 [  64    0 1265]]


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize

# 6. Dự đoán

In [15]:
def my_prediction(new_sentence):
    # 1. Biến câu chữ thành Vector số 
    vector_cau_moi = vectorizer.transform([new_sentence])
    
    # 2. Dự đoán nhãn (0, 1, hoặc 2)
    y_pred = model.predict(vector_cau_moi)[0]
    
    # 3. Xem xác suất phần trăm cho từng lớp 
    prob = model.predict_proba(vector_cau_moi)[0]
    
    # Mapping số về chữ
    labels_map = {0: "Tiêu cực (Negative)", 1: "Trung tính (Neutral)", 2: "Tích cực (Positive)"}
    
    print(f"\n--- KẾT QUẢ DỰ ĐOÁN ---")
    print(f"Câu của tôi: '{new_sentence}'")
    print(f"Dự đoán: {labels_map[y_pred]}")
    print(f"Độ tự tin: Neg: {prob[0]:.2f}, Neu: {prob[1]:.2f}, Pos: {prob[2]:.2f}")

# --- TEST THỬ ---
ex1 = "nhân viên cọc tính, đồ ăn phục vụ khá lâu và nó dở, nhà hàng không sang sọng, tôi sẽ không ăn ở đây thêm lần nữa."
ex2='nói chung cũng bình thường'
my_prediction(ex1)
my_prediction(ex2)
my_prediction(input("Nhập câu review của bạn: "))


--- KẾT QUẢ DỰ ĐOÁN ---
Câu của tôi: 'nhân viên cọc tính, đồ ăn phục vụ khá lâu và nó dở, nhà hàng không sang sọng, tôi sẽ không ăn ở đây thêm lần nữa.'
Dự đoán: Tiêu cực (Negative)
Độ tự tin: Neg: 0.67, Neu: 0.01, Pos: 0.32

--- KẾT QUẢ DỰ ĐOÁN ---
Câu của tôi: 'nói chung cũng bình thường'
Dự đoán: Trung tính (Neutral)
Độ tự tin: Neg: 0.20, Neu: 0.54, Pos: 0.26

--- KẾT QUẢ DỰ ĐOÁN ---
Câu của tôi: 'huhu'
Dự đoán: Tích cực (Positive)
Độ tự tin: Neg: 0.20, Neu: 0.03, Pos: 0.77
